<font>
<div dir=ltr align=center>
<img src='https://cdn.freebiesupply.com/logos/large/2x/sharif-logo-png-transparent.png' width=150 height=150> <br>
<font color=0F5298 size=6>
Machine Learning<br>
<font color=2565AE size=4>
Computer Engineering Department<br>
Fall 2025<br>
<font color=3C99D size=4>
Homework 3 - Practical<br>
<font color=696880 size=3>

# Libraries

In [ ]:
import math
import time
import copy
import numpy as np
import matplotlib.pyplot as plt

# Part 1: Dataset

In [ ]:
def set_seed(seed=42):
    """Return a NumPy Generator for reproducibility."""
    generator = np.random.default_rng(seed)
    return generator


def train_val_split(X, y, val_ratio=0.2, seed=42):
    """Split X, y into train/val using a shuffled index."""
    generator = set_seed(seed)
    n = X.shape[0]
    idx = np.arange(n)
    generator.shuffle(idx)

    split_idx = int(n * (1 - val_ratio))
    train_idx, val_idx = idx[:split_idx], idx[split_idx:]
    
    return X[train_idx], y[train_idx], X[val_idx], y[val_idx]

def standardize(X_train, X_val):
    """Standardize features using train mean/std and apply to val."""
    mean = X_train.mean(axis=0)
    std = X_train.std(axis=0) + 1e-12
    X_train_norm = (X_train - mean) / std
    X_val_norm = (X_val - mean) / std
    return X_train_norm, X_val_norm

In [ ]:
def make_blobs(n_samples=1000, n_features=2, centers=3, cluster_std=1.0, seed=42):
    """Generate a simple clustered dataset (features X, integer labels y)."""
    generator = set_seed(seed)
    X = []
    y = []
    
    # Generate random centers
    centers_coords = generator.uniform(-10, 10, size=(centers, n_features))
    
    samples_per_center = n_samples // centers
    for i in range(centers):
        cluster_X = centers_coords[i] + generator.normal(0, cluster_std, size=(samples_per_center, n_features))
        X.append(cluster_X)
        y.append(np.full(samples_per_center, i))
        
    return np.vstack(X), np.concatenate(y)

In [ ]:
X, y = make_blobs(n_samples=1200, n_features=10, centers=5, cluster_std=1.3)
X_train_raw, y_train_raw, X_val_raw, y_val_raw = train_val_split(X, y, val_ratio=0.2)
X_train, X_val = standardize(X_train_raw, X_val_raw)
y_train, y_val = y_train_raw, y_val_raw

In [ ]:
num_classes = int(y.max() + 1)
input_dim = X_train.shape[1]
print('Train shape:', X_train.shape, 'Val shape:', X_val.shape, 'Classes:', num_classes)

# Part 2: MLP Layers

In [ ]:
class Layer:
    def __init__(self):
        self.params = {}
        self.grads = {}

    def forward(self, x, training=True):
        raise NotImplementedError()

    def backward(self, dout):
        raise NotImplementedError()

    def parameters(self):
        for k, v in self.params.items():
            yield k, v, self.grads[k]

In [ ]:
class Linear(Layer):
    def __init__(self, in_features, out_features, init='he', seed=42):
        super().__init__()
        if init == 'xavier':
            scale = math.sqrt(2.0 / (in_features + out_features))
        else:
            scale = math.sqrt(2.0 / in_features)

        generator = np.random.default_rng(seed)
        self.params['W'] = generator.normal(0.0, scale, size=(in_features, out_features))
        self.params['b'] = np.zeros((out_features,))
        self.grads['W'] = np.zeros_like(self.params['W'])
        self.grads['b'] = np.zeros_like(self.params['b'])
        self.cache = None

    def forward(self, x, training=True):
        self.cache = x
        return x @ self.params['W'] + self.params['b']

    def backward(self, dout):
        x = self.cache
        self.grads['W'] = x.T @ dout
        self.grads['b'] = np.sum(dout, axis=0)
        dx = dout @ self.params['W'].T
        return dx

In [ ]:
class ReLU(Layer):
    def __init__(self):
        super().__init__()
        self.mask = None

    def forward(self, x, training=True):
        self.mask = (x > 0)
        return x * self.mask

    def backward(self, dout):
        return dout * self.mask

class Tanh(Layer):
    def __init__(self):
        super().__init__()
        self.cache = None

    def forward(self, x, training=True):
        out = np.tanh(x)
        self.cache = out
        return out

    def backward(self, dout):
        out = self.cache
        return dout * (1 - out * out)

In [ ]:
class Dropout(Layer):
    def __init__(self, p=0.5, seed=42):
        super().__init__()
        self.p = float(p)
        self.keep = 1.0 - self.p
        self.generator = np.random.default_rng(seed)
        self.mask = None

    def forward(self, x, training=True):
        if not training or self.p == 0.0:
            self.mask = None
            return x
        self.mask = (self.generator.random(x.shape) < self.keep).astype(np.float64) / self.keep
        return x * self.mask

    def backward(self, dout):
        if self.mask is None:
            return dout
        return dout * self.mask

In [ ]:
class BatchNorm1d(Layer):
    def __init__(self, num_features, momentum=0.9, eps=1e-5):
        super().__init__()
        self.params['gamma'] = np.ones((num_features,))
        self.params['beta'] = np.zeros((num_features,))
        self.grads['gamma'] = np.zeros_like(self.params['gamma'])
        self.grads['beta'] = np.zeros_like(self.params['beta'])
        self.momentum = momentum
        self.eps = eps
        self.running_mean = np.zeros((num_features,))
        self.running_var = np.ones((num_features,))
        self.cache = None

    def forward(self, x, training=True):
        gamma, beta = self.params['gamma'], self.params['beta']
        if training:
            mu = x.mean(axis=0)
            var = x.var(axis=0)
            x_mu = x - mu
            std_inv = 1.0 / np.sqrt(var + self.eps)
            x_hat = x_mu * std_inv
            self.running_mean = self.momentum * self.running_mean + (1 - self.momentum) * mu
            self.running_var = self.momentum * self.running_var + (1 - self.momentum) * var
            self.cache = (x, x_hat, mu, var, std_inv)
        else:
            x_hat = (x - self.running_mean) / np.sqrt(self.running_var + self.eps)
        return gamma * x_hat + beta

    def backward(self, dout):
        x, x_hat, mu, var, std_inv = self.cache
        gamma = self.params['gamma']
        N, D = x.shape
        self.grads['gamma'] = np.sum(dout * x_hat, axis=0)
        self.grads['beta'] = np.sum(dout, axis=0)
        dx_hat = dout * gamma
        dvar = np.sum(dx_hat * (x - mu), axis=0) * (-0.5) * (std_inv ** 3)
        dmu = np.sum(dx_hat * -std_inv, axis=0) + dvar * np.mean(-2 * (x - mu), axis=0)
        dx = dx_hat * std_inv + dvar * 2 * (x - mu) / N + dmu / N
        return dx

# Part 3: Loss

In [ ]:
def softmax_cross_entropy(logits, y):
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp = np.exp(shifted)
    probs = exp / np.sum(exp, axis=1, keepdims=True)
    N = logits.shape[0]
    loss = -np.log(probs[np.arange(N), y] + 1e-12).mean()
    dlogits = probs.copy()
    dlogits[np.arange(N), y] -= 1
    dlogits /= N
    return loss, dlogits

# Part 4: MLP

In [ ]:
class MLP:
    def __init__(self, input_dim, hidden_sizes, num_classes, activation='relu', use_batchnorm=False, dropout_p=0.0, init='he', seed=42):
        self.layers = []
        rng = np.random.default_rng(seed)
        
        dims = [input_dim] + hidden_sizes + [num_classes]
        for i in range(len(dims) - 1):
            layer_seed = int(rng.integers(0, 1000000))
            self.layers.append(Linear(dims[i], dims[i+1], init=init, seed=layer_seed))
            
            if i < len(dims) - 2: # No BN/Act/Dropout on output layer
                if use_batchnorm:
                    self.layers.append(BatchNorm1d(dims[i+1]))
                
                if activation == 'relu':
                    self.layers.append(ReLU())
                else:
                    # Could add Tanh here if needed
                    pass
                
                if dropout_p > 0:
                    self.layers.append(Dropout(p=dropout_p, seed=layer_seed+1))

    def forward(self, x, training=True):
        for layer in self.layers:
            x = layer.forward(x, training=training)
        return x

    def backward(self, dout):
        for layer in reversed(self.layers):
            dout = layer.backward(dout)

    def zero_grads(self):
        for layer in self.layers:
            for g in layer.grads.values():
                g.fill(0.0)

    def parameters(self):
        for layer in self.layers:
            for name, param in layer.params.items():
                yield id(param), name, param, layer.grads[name], layer

    def predict(self, x):
        logits = self.forward(x, training=False)
        return np.argmax(logits, axis=1)

    def accuracy(self, x, y):
        return np.mean(self.predict(x) == y)

## Numerical gradient check (sanity)
We compare analytical gradients vs. finite differences on a tiny network and batch.

In [ ]:
def numerical_grad_check(model, X_batch, y_batch, eps=1e-5, atol=1e-6, rtol=1e-3, verbose=False):
    # TODO 1: Forward pass
    logits = model.forward(X_batch, training=True)

    # TODO 2: Loss and analytic grads
    loss, dout = softmax_cross_entropy(logits, y_batch)

    # TODO 3: Zero grads
    model.zero_grads()

    # TODO 4: Backward pass
    model.backward(dout)

    # TODO 5: Flag
    ok = True

    # TODO 6: Loop over parameters
    for pid, name, param, grad, layer in model.parameters():
        if param is None or grad is None:
            continue
        
        flat = param.reshape(-1)
        gflat = grad.reshape(-1)
        
        rng = np.random.default_rng(0)
        idxs = rng.choice(len(flat), size=min(5, len(flat)), replace=False)

        # TODO 7: Numerical estimation
        for idx in idxs:
            old = flat[idx]
            
            flat[idx] = old + eps
            logits_pos = model.forward(X_batch, training=True)
            loss_pos, _ = softmax_cross_entropy(logits_pos, y_batch)
            
            flat[idx] = old - eps
            logits_neg = model.forward(X_batch, training=True)
            loss_neg, _ = softmax_cross_entropy(logits_neg, y_batch)
            
            flat[idx] = old
            g_num = (loss_pos - loss_neg) / (2 * eps)
            g_ana = gflat[idx]

            # TODO 8: Comparison
            if not np.isclose(g_num, g_ana, atol=atol, rtol=rtol):
                ok = False
                if verbose:
                    print(f"Grad mismatch in {layer.__class__.__name__}.{name}: "
                          f"numeric={g_num:.6e}, analytic={g_ana:.6e}")
    
    return ok

In [ ]:
# Run a quick check (no dropout during grad check)
Xb = X_train[:8].astype(np.float64)
yb = y_train[:8]

tiny = MLP(input_dim=input_dim, hidden_sizes=[8], num_classes=num_classes,
           activation='relu', use_batchnorm=True, dropout_p=0.0, init='he', seed=123)

ok = numerical_grad_check(tiny, Xb, yb, eps=1e-6, atol=1e-8, rtol=1e-5, verbose=True)

print('Gradient check passed?', ok)

# Part 4: Optimizers
We implement: SGD, Momentum, Nesterov, RMSProp, Adam, AdamW (decoupled weight decay).

In [ ]:
# سلول ۲۲
class Optimizer:
    def __init__(self, lr=1e-2):
        self.lr = lr
        self.state = {}

    def _get_state(self, pid):
        if pid not in self.state:
            self.state[pid] = {}
        return self.state[pid]

    def step(self, params_and_grads):
        raise NotImplementedError()

In [ ]:
# سلول ۲۳ (SGD)
class SGD(Optimizer):
    def step(self, params_and_grads):
        for pid, name, param, grad, layer in params_and_grads:
            param[...] = param - self.lr * grad

In [ ]:
# سلول ۲۴ (SGDMomentum)
class SGDMomentum(Optimizer):
    def __init__(self, lr=1e-2, momentum=0.9):
        super().__init__(lr)
        self.momentum = momentum

    def step(self, params_and_grads):
        for pid, name, param, grad, layer in params_and_grads:
            state = self._get_state(pid)
            if 'v' not in state:
                state['v'] = np.zeros_like(param)
            
            state['v'] = self.momentum * state['v'] - self.lr * grad
            param[...] = param + state['v']

In [ ]:
# سلول ۲۵ (Nesterov)
class Nesterov(Optimizer):
    def __init__(self, lr=1e-2, momentum=0.9):
        super().__init__(lr)
        self.momentum = momentum

    def step(self, params_and_grads):
        for pid, name, param, grad, layer in params_and_grads:
            state = self._get_state(pid)
            if 'v' not in state:
                state['v'] = np.zeros_like(param)
            
            v_prev = state['v'].copy()
            state['v'] = self.momentum * state['v'] - self.lr * grad
            param[...] = param + (-self.momentum * v_prev + (1 + self.momentum) * state['v'])

In [ ]:
# سلول ۲۶ (RMSProp)
class RMSProp(Optimizer):
    def __init__(self, lr=1e-3, rho=0.9, eps=1e-8):
        super().__init__(lr)
        self.rho = rho
        self.eps = eps

    def step(self, params_and_grads):
        for pid, name, param, grad, layer in params_and_grads:
            state = self._get_state(pid)
            if 's' not in state:
                state['s'] = np.zeros_like(param)
            
            state['s'] = self.rho * state['s'] + (1 - self.rho) * (grad**2)
            param[...] = param - self.lr * grad / (np.sqrt(state['s']) + self.eps)

In [ ]:
# سلول ۲۷ (Adam)
class Adam(Optimizer):
    def __init__(self, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0, decoupled=False):
        super().__init__(lr)
        self.beta1, self.beta2, self.eps = beta1, beta2, eps
        self.wd, self.decoupled = weight_decay, decoupled
        self.t = 0

    def step(self, params_and_grads):
        self.t += 1
        for pid, name, param, grad, layer in params_and_grads:
            state = self._get_state(pid)
            if 'm' not in state:
                state['m'] = np.zeros_like(param)
                state['v'] = np.zeros_like(param)
            
            g = grad.copy()
            if self.wd > 0:
                if self.decoupled:
                    param[...] -= self.lr * self.wd * param
                else:
                    g += self.wd * param
            
            state['m'] = self.beta1 * state['m'] + (1 - self.beta1) * g
            state['v'] = self.beta2 * state['v'] + (1 - self.beta2) * (g**2)
            
            m_hat = state['m'] / (1 - self.beta1**self.t)
            v_hat = state['v'] / (1 - self.beta2**self.t)
            
            param[...] -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Part 5: L1/L2 Regularization

In [ ]:
# سلول ۲۹
def apply_weight_regularization(model, l1=0.0, l2=0.0):
    reg_loss = 0.0
    for layer in model.layers:
        if isinstance(layer, Linear):
            W = layer.params['W']
            reg_loss += l1 * np.sum(np.abs(W)) + 0.5 * l2 * np.sum(W**2)
            layer.grads['W'] += l1 * np.sign(W) + l2 * W
    return reg_loss

# Part 6: Training Loop

In [ ]:
# سلول ۳۱
def iterate_minibatches(X, y, batch_size, shuffle=True, seed=42):
    n = X.shape[0]
    idx = np.arange(n)
    if shuffle:
        set_seed(seed).shuffle(idx)
    
    for i in range(0, n, batch_size):
        yield X[idx[i:i+batch_size]], y[idx[i:i+batch_size]]

In [ ]:
# سلول ۳۲
def train(model, optimizer, X_train, y_train, X_val, y_val,
          epochs=30, batch_size=64, l1=0.0, l2=0.0,
          early_stopping_patience=5, seed=42, verbose=True,
          micro_batch_size=None):
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    patience_cnt = 0
    
    for epoch in range(1, epochs + 1):
        t_losses, t_accs = [], []
        for Xb, yb in iterate_minibatches(X_train, y_train, batch_size, seed=seed+epoch):
            model.zero_grads()
            
            if micro_batch_size and micro_batch_size < batch_size:
                batch_loss = 0
                for i in range(0, batch_size, micro_batch_size):
                    Xm, ym = Xb[i:i+micro_batch_size], yb[i:i+micro_batch_size]
                    logits_m = model.forward(Xm, training=True)
                    loss_m, dlogits_m = softmax_cross_entropy(logits_m, ym)
                    # Scale grads for accumulation
                    dlogits_m *= (len(Xm) / batch_size)
                    model.backward(dlogits_m)
                    batch_loss += loss_m * (len(Xm) / batch_size)
                loss = batch_loss
            else:
                logits = model.forward(Xb, training=True)
                loss, dlogits = softmax_cross_entropy(logits, yb)
                model.backward(dlogits)
            
            reg_loss = apply_weight_regularization(model, l1, l2)
            loss += reg_loss
            optimizer.step(list(model.parameters()))
            
            t_losses.append(loss)
            t_accs.append(np.mean(np.argmax(model.forward(Xb, training=False), axis=1) == yb))
            
        val_logits = model.forward(X_val, training=False)
        val_loss, _ = softmax_cross_entropy(val_logits, y_val)
        val_acc = model.accuracy(X_val, y_val)
        
        history['train_loss'].append(np.mean(t_losses))
        history['train_acc'].append(np.mean(t_accs))
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        if verbose:
            print(f"Epoch {epoch}: t_loss={history['train_loss'][-1]:.4f}, v_acc={val_acc:.3f}")
            
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_cnt = 0
        else:
            patience_cnt += 1
            if patience_cnt >= early_stopping_patience:
                break
    return history

In [ ]:
model = MLP(
    input_dim=X_train.shape[1],
    hidden_sizes=[64, 64],
    num_classes=int(y_train.max() + 1),
    activation='relu',
    use_batchnorm=False,
    dropout_p=0.0,
    init='he',
    seed=123,
)

optimizer = Adam(lr=1e-3)

history = train(model, optimizer, X_train, y_train, X_val, y_val,
    epochs=80, batch_size=64, l1=0.0, l2=0.0, early_stopping_patience=12, seed=1, verbose=True, micro_batch_size=None)

val_acc = model.accuracy(X_val, y_val)
print('Final val_acc =', val_acc)

assert val_acc >= 0.99, 'Expected ~100% val accuracy on the blobs sanity run.'

# Experiments

In [ ]:
def plot_metric(history_list, labels, key, title):
    plt.figure(figsize=(6,4))
    for history, lab in zip(history_list, labels):
        plt.plot(history[key], label=lab)

    plt.title(title)
    plt.xlabel('Epoch')
    plt.ylabel(key.replace('_', ' '))
    plt.legend()
    plt.show()

## Experiment A: Optimizer comparison (same budget)
We fix the model/seed and compare optimizers on the synthetic blobs (no dropout or L1/L2 to keep it clean).

In [ ]:
histories = []
labels = []

for optimizer_name, optimizer in [
    ('SGD', SGD(lr=1e-1)),
    ('Momentum', SGDMomentum(lr=5e-2, momentum=0.9)),
    ('Nesterov', Nesterov(lr=5e-2, momentum=0.9)),
    ('RMSProp', RMSProp(lr=1e-3, rho=0.9)),
    ('Adam', Adam(lr=1e-3)),
    ('AdamW', Adam(lr=1e-3, weight_decay=1e-2, decoupled=True)),
]:
    model = MLP(input_dim=input_dim, hidden_sizes=[64, 64], num_classes=num_classes,
                activation='relu', use_batchnorm=False, dropout_p=0.0, init='he', seed=123)
    history = train(model, optimizer, X_train, y_train, X_val, y_val,
                    epochs=40, batch_size=64, l1=0.0, l2=0.0, early_stopping_patience=5, seed=123, verbose=False)

    print(f'{optimizer_name:8s} final val acc: {model.accuracy(X_val, y_val):.3f}')

    histories.append(history)
    labels.append(optimizer_name)

In [ ]:
plot_metric(histories, labels, 'val_loss', 'Validation Loss by Optimizer')
plot_metric(histories, labels, 'val_acc', 'Validation Accuracy by Optimizer')

## Experiment B: Regularization ablation (L1/L2/Dropout)
We test None vs. **L2** vs. **L1** vs. **Dropout (p=0.5)**. Keep optimizer fixed for fairness.

In [ ]:
histories = []
labels = []

for l1, l2, dropout_p, name in [
    (0.0, 0.0, 0.0, 'none'),
    (0.0, 1e-3, 0.0, 'L2=1e-3'),
    (1e-4, 0.0, 0.0, 'L1=1e-4'),
    (0.0, 0.0, 0.5, 'Dropout p=0.5'),
]:
    model = MLP(input_dim=input_dim, hidden_sizes=[64, 64], num_classes=num_classes,
                activation='relu', use_batchnorm=False, dropout_p=dropout_p, init='he', seed=123)
    opt = SGDMomentum(lr=5e-2, momentum=0.9)
    history = train(model, opt, X_train, y_train, X_val, y_val,
                 epochs=40, batch_size=64, l1=l1, l2=l2, early_stopping_patience=5, seed=123, verbose=False)
    val_acc = model.accuracy(X_val, y_val)
    print(f'{name:15s} | val_acc={val_acc:.3f}')

    histories.append(history)
    labels.append(name)

In [ ]:
plot_metric(histories, labels, 'val_loss', 'Validation Loss (Regularization)')
plot_metric(histories, labels, 'val_acc', 'Validation Accuracy (Regularization)')

## Experiment C: BatchNorm on/off (with recommended ordering)
Compare MLP **with** vs. **without** BatchNorm (Linear -> BN -> ReLU -> [Dropout]). Keep optimizer and budget fixed.

In [ ]:
def run_batch_norm(use_batch_norm, name):
    model = MLP(input_dim=input_dim, hidden_sizes=[64, 64], num_classes=num_classes,
                activation='relu', use_batchnorm=use_batch_norm, dropout_p=0.3, init='he', seed=123)
    optimizer = SGDMomentum(lr=5e-2, momentum=0.9)
    history = train(model, optimizer, X_train, y_train, X_val, y_val,
                 epochs=40, batch_size=64, l1=0.0, l2=0.0, early_stopping_patience=5, seed=123, verbose=False)
    val_acc = model.accuracy(X_val, y_val)

    print(f'{name:8s} | val_acc={val_acc:.3f}')
    return history, name

batch_norm_history, _ = run_batch_norm(True, 'with BN')
no_batch_norm_history, _ = run_batch_norm(False, 'no BN')

In [ ]:
plot_metric([batch_norm_history, no_batch_norm_history], ['with BN', 'no BN'], 'val_loss', 'Validation Loss (BN vs No BN)')
plot_metric([batch_norm_history, no_batch_norm_history], ['with BN', 'no BN'], 'val_acc', 'Validation Accuracy (BN vs No BN)')

## Experiment C: Micro-batch ablation (Gradient Accumulation)
Keep optimizer fixed for fairness; hold logical batch_size constant
and vary micro_batch_size ∈ {None, 32, 16, 8, 4, 1}

In [ ]:
histories_mb = []
labels_mb = []

logical_batch_size = 64   
optimizer_ctor = lambda: SGDMomentum(lr=5e-2, momentum=0.9)  

for micro_bs, name in [
    (None, 'no-accum'),       
    (32,   'micro=32 (x2)'),   
    (16,   'micro=16 (x4)'),
    (8,    'micro=8  (x8)'),
    (4,    'micro=4  (x16)'),
    (1,    'micro=1  (x64)'),
]:
    model = MLP(input_dim=input_dim, hidden_sizes=[64, 64], num_classes=num_classes,
                activation='relu', use_batchnorm=False, dropout_p=0.0, init='he', seed=123)

    opt = optimizer_ctor()

    history = train(model, opt, X_train, y_train, X_val, y_val,
                    epochs=40,
                    batch_size=logical_batch_size,      
                    l1=0.0, l2=0.0,
                    early_stopping_patience=5,
                    seed=123,
                    verbose=False,
                    micro_batch_size=micro_bs)           

    val_acc = model.accuracy(X_val, y_val)
    print(f'{name:12s} | val_acc={val_acc:.3f}')

    histories_mb.append(history)
    labels_mb.append(name)

# Plot like before
plot_metric(histories_mb, labels_mb, 'val_loss', 'Validation Loss by Micro-batch Size')
plot_metric(histories_mb, labels_mb, 'val_acc',  'Validation Accuracy by Micro-batch Size')